# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring the FAIR² dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

We will print all available record sets, their `@id`, and their fields with respective `@id`s.

In [ ]:
record_sets = list(dataset.record_sets)

print("Number of record sets:", len(record_sets))

for rs in record_sets:
    print(f"\nRecordSet name: {rs.name}")
    print(f"  @id: {rs.id}")
    field_ids = []
    for f in rs.fields:
        if hasattr(f, 'id'):
            field_ids.append(f.id)
            print(f"    Field: {getattr(f, 'name', 'N/A')}, @id: {f.id}, dataType: {getattr(f, 'data_type', 'N/A')}")
    if hasattr(rs, 'columns'):
        for c in rs.columns:
            print(f"    Column: {getattr(c, 'name', 'N/A')}, @id: {c.id}, dataType: {getattr(c, 'data_type', 'N/A')}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview above.

Below, we load all record sets and show the first DataFrame's structure.

In [ ]:
# Identify record set ids
record_set_ids = [rs.id for rs in dataset.record_sets]
dataframes = {}
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df

# Show one example
if record_set_ids:
    example_set = record_set_ids[0]
    print(f'Example Record Set: {example_set}')
    print('Columns:', dataframes[example_set].columns.tolist())
    display(dataframes[example_set].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section may include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

Below, we select a numeric field and a group/categorical field by their `@id`. If available, these will be used for demonstration.

In [ ]:
# Choose record set and field IDs (change according to data overview above)
selected_record_set = example_set
df = dataframes[selected_record_set]

# Attempt automatic selection of a numeric field and a group field
numeric_field = None
group_field = None

# Find a float or integer column
for col in df.columns:
    try:
        # Check type by attempting conversion
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_field = col
            break
    except Exception:
        continue
        
# Find a non-numeric field for groups
for col in df.columns:
    if col != numeric_field and df[col].dtype == object:
        group_field = col
        break

if numeric_field:
    threshold = df[numeric_field].quantile(0.5)  # Use median for demonstration
    filtered_df = df[df[numeric_field] > threshold].copy()
    print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
    display(filtered_df.head())

    # Normalize numeric field
    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"Normalized {numeric_field} for filtered records:")
    display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    if group_field and group_field in df.columns:
        grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
        print(f"Grouped data by {group_field}:")
        display(grouped_df.head())
else:
    print('No numeric field found in this record set for EDA.')

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.
The example below plots the distribution of the chosen numeric field, if possible.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field and not filtered_df.empty:
    plt.figure(figsize=(8,4))
    sns.histplot(filtered_df[numeric_field], kde=True)
    plt.title(f'Distribution of {numeric_field}')
    plt.xlabel(numeric_field)
    plt.ylabel('Count')
    plt.show()
    
    if group_field:
        plt.figure(figsize=(10, 5))
        sns.boxplot(x=group_field, y=numeric_field, data=filtered_df)
        plt.title(f'{numeric_field} by {group_field}')
        plt.xticks(rotation=45)
        plt.show()
else:
    print('Visualization not possible: No numeric field or empty filtered dataframe.')

## 6. Conclusion
In this notebook, we demonstrated the use of the `mlcroissant` library to load, inspect, and analyze datasets defined by a Croissant schema. This process included reviewing the dataset structure, extracting records, processing numeric and categorical fields, and visualizing the results for the FAIR² Mass Spectrometry dataset.

For more advanced analysis, further data curation, field selection by `@id`, and domain knowledge can be combined with the reproducible workflow provided here.